# SAF Cacau Bahia — logística rodoviária até Ilhéus

Notebook gratuito para calcular distância viária entre os centros dos hexágonos/imóveis certificados e uma unidade de destino em Ilhéus.

**Método:** malha OpenStreetMap Nordeste + menor caminho em grafo dirigido. O resultado é uma triagem logística regional; não substitui cotação de frete, inspeção da via ou validação de restrições de carreta.

In [ ]:
# Dependências do ambiente temporário Colab
!pip -q install pyrosm networkx geopandas pyarrow scipy

In [ ]:
from google.colab import files
uploaded = files.upload()
# Envie estes dois arquivos do projeto:
# 1) data/processed/bahia_hex_5km.geojson
# 2) data/processed/fazendas_georeferenciadas.geoparquet

In [ ]:
# CONFIGURAÇÃO — preencher com a coordenada confirmada do portão/unidade Cargill.
# A referência documental é Rodovia Ilhéus–Uruçuca, Distrito Industrial/Iguape.
# Não use o centro do município.
DESTINO_LAT = None
DESTINO_LON = None
DESTINO_NOME = 'Cargill Cacau — Ilhéus/BA'

assert DESTINO_LAT is not None and DESTINO_LON is not None, 'Informe a coordenada confirmada do destino.'
OSM_URL = 'https://download.geofabrik.de/south-america/brazil/nordeste-latest.osm.pbf'
OSM_FILE = 'nordeste-latest.osm.pbf'
ANALYSIS_CRS = 'EPSG:5880'

In [ ]:
# Download temporário da malha rodoviária. O arquivo será apagado ao final.
!wget -q --show-progress -O {OSM_FILE} {OSM_URL}
!ls -lh {OSM_FILE}

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from scipy.spatial import cKDTree
from pyrosm import OSM

hex_path = next(Path('.').glob('bahia_hex_5km.geojson'))
farm_path = next(Path('.').glob('fazendas_georeferenciadas.geoparquet'))
hexes = gpd.read_file(hex_path)
farms = gpd.read_parquet(farm_path)
assert hexes['id_hex'].is_unique and farms['id_fazenda'].is_unique
print(f'Hexágonos: {len(hexes):,}; imóveis: {len(farms):,}')

In [ ]:
# Rede dirigida para veículos. OSM fornece as direções e comprimento dos segmentos.
osm = OSM(OSM_FILE)
nodes, edges = osm.get_network(network_type='driving', nodes=True)
nodes = nodes[nodes.geometry.notna()].copy()
edges = edges[edges['u'].notna() & edges['v'].notna()].copy()
if 'length' not in edges.columns:
    edges_metric = edges.to_crs(ANALYSIS_CRS)
    edges['length'] = edges_metric.geometry.length

# Pyrosm retorna arcos dirigidos adequados ao perfil driving.
graph = nx.DiGraph()
graph.add_weighted_edges_from(zip(edges['u'].astype(str), edges['v'].astype(str), edges['length'].astype(float)))
print(f'Nós: {graph.number_of_nodes():,}; arcos: {graph.number_of_edges():,}')

In [ ]:
# Índice de nós em CRS métrico para associar cada centroide à rede.
node_id_col = 'id' if 'id' in nodes.columns else nodes.columns[0]
nodes = nodes[nodes[node_id_col].astype(str).isin(graph.nodes)].copy()
nodes_metric = nodes.to_crs(ANALYSIS_CRS)
node_ids = nodes_metric[node_id_col].astype(str).to_numpy()
node_xy = np.c_[nodes_metric.geometry.x.to_numpy(), nodes_metric.geometry.y.to_numpy()]
tree = cKDTree(node_xy)

destination = gpd.GeoSeries(gpd.points_from_xy([DESTINO_LON], [DESTINO_LAT]), crs='EPSG:4326').to_crs(ANALYSIS_CRS)
_, destination_pos = tree.query([[destination.x.iloc[0], destination.y.iloc[0]]], k=1)
destination_node = node_ids[int(destination_pos[0])]

# Um único Dijkstra reverso obtém a menor distância de cada nó até Ilhéus.
distance_to_ilheus_m = nx.single_source_dijkstra_path_length(graph.reverse(copy=False), destination_node, weight='weight')
print(f'Nós com rota até o destino: {len(distance_to_ilheus_m):,}')

In [ ]:
def route_results(frame, id_field):
    # Centro geométrico, conforme regra definida para hexágonos e imóveis.
    metric = frame.to_crs(ANALYSIS_CRS).copy()
    centres = metric.geometry.centroid
    xy = np.c_[centres.x.to_numpy(), centres.y.to_numpy()]
    snap_m, pos = tree.query(xy, k=1)
    snapped_ids = node_ids[pos]
    network_m = np.array([distance_to_ilheus_m.get(node, np.nan) for node in snapped_ids], dtype=float)
    total_m = network_m + snap_m
    centres_wgs = gpd.GeoSeries(centres, crs=ANALYSIS_CRS).to_crs('EPSG:4326')
    return pd.DataFrame({
        id_field: frame[id_field].astype(str).to_numpy(),
        'logistica_origem_lon': centres_wgs.x,
        'logistica_origem_lat': centres_wgs.y,
        'logistica_distancia_rota_km': total_m / 1000,
        'logistica_distancia_snap_km': snap_m / 1000,
        'logistica_status_rota': np.where(np.isfinite(network_m), 'ROTA_OSM_CALCULADA', 'SEM_ROTA_NA_REDE'),
        'logistica_fonte_rota': 'OSM_GEOFABRIK_NORDESTE+NETWORKX_DIJKSTRA_REVERSO',
        'logistica_destino': DESTINO_NOME,
    })

hex_result = route_results(hexes, 'id_hex')
farm_result = route_results(farms, 'id_fazenda')
print(hex_result['logistica_status_rota'].value_counts(dropna=False))
print(farm_result['logistica_status_rota'].value_counts(dropna=False))

In [ ]:
# Exportar somente os resultados leves. A nota de logística será calculada no projeto principal.
hex_result.to_parquet('logistica_hex_resultados.parquet', index=False)
farm_result.to_parquet('logistica_imoveis_resultados.parquet', index=False)
files.download('logistica_hex_resultados.parquet')
files.download('logistica_imoveis_resultados.parquet')

# Limpeza opcional da malha temporária do Colab
# os.remove(OSM_FILE)